# EuromonitoR

### Entity Matching - So What, Who Cares?

Business Case:  
Wrong metrics will invalidate all downstream economic studies. 
  
    
`Economical inaccuracies on my watch? Think again.`


  ## Core problem

Same physical product with GTIN (barcode/ Ground Truth) but many are missing/invalid/reused, one GTIN  
can carry inconsistent attributes across retailers, and different GTINs can describe the same product. 

  
    
    Therefore: Given a SKU and a candidate canonical product, predict whether they represent the same physical product / GTIN.

# Looks like A GTIN, But is it?
 This dataset required more of a judgment call rather than anything else:
   - Define Ground Truth  
   - Model Input
   - Loss Function
   - Model Architecture
   - Outcomes

### My Definition:
    1. **GOLD+** — valid GTIN, >= 2 retailers: the only positives (17,909 pairs).
    2. **SINGLE** — valid GTIN, 1 retailer (7,000 rows): never a positive.
    3. **SUSPECT** — same title, different valid GTIN (228 groups) [Fallback to CrossEncoder]
    4. **UNLABELED** — missing / invalid GTIN (37k rows): pseudo-label territory only. 

   Positives are stratified (easy >= 0.8 / hard < 0.5 jaccard) 

### Model Input
   - Multiple Negatives
   - Hard +/-
   - Anchor +/-
   - SKU / SKU Same/Diff GTIN
   - Canonical / Canonical
   - SKU-to-canonical

### Loss Functions
Candidates:
   - Online Contrastive Learning
   - Triplet
   - CosineSim
   - MarginalRank 
   
### Loss Functions
   - Two Tower / Bi-Encoder
   - Cross Encoder
   - Both  


### Outcomes
   - SKU-Canonical Matching
   - General Entity Representation

### In this Project
Since 




                           Raw SKU  
                              ↓  
                        NER / normalization  
                              ↓  
                        volume / pack / flavor / attributes  
                              ↓  
                        deterministic gate  
                              ↓  
                        clean semantic text  
                              ↓  
                        Two-Tower embedding  


## Approach
1. **Validate GTINs** (length, check digit) — clean vs noisy barcodes.
2. **Extract NER** For Vol, attributes +, brand name, de-noise and produce labels.
3. **Deterministic three-way gate** on volume/pack/flavor: block impossible
   matches (hard_no), route uncertain ones to fallback, send likely
   duplicates to embeddings (proceed).
4. **Fine-tune an embedding model** on cleaned text (NO numbers — sizes are
   the gate's job, never the model's) to learn product identity.
5. **Component-fold evaluation** — no barcode straddles a split boundary;
   metrics are honest (PR-AUC primary, F1 at a fixed threshold) on holdout set


# GTIN
`Pandas`
- Develop Trusted GT: GTIN 58% missing; of the 42% populated, 43.1% are non-unique; 0.6% shared across different brands .`. Untrustworthy.   
- GTIN imbalance was around 1:17, with masking[1:1] on agg data, resulting in an almost 1:1. 
- Iimilar descriptions but different products (hard negatives) serve for OnlineContrastiveLoss, Tripletloss, and MultipleNegativesRakingLoss.     
  
### TODO
- Verify masking[%] effect

# NER [Pack, Flavor, Vol, Attr] 
`TfidfVectorizer, cosine_similarity, fuzzymatching, regex`  

- Create better representations to generalize for general future use.
- Volume: generally present (92%+) BUT the most complicated to clean .`.  Gate that pairs similar products as model payload.
- Brand: Cleaning `sku_name_eng` meant removing brand + edge cases.



# HPO
`Optuna MLFlow, Kubernetes`  
ObjectiveF: mean best development-set average precision

## Conclusion
- Previous submission had plenty of problems, data leakage and dirty model payload. `FIX` Group-aware/component split to prevent entity leakage.




# Coding Conventions:
- SSOT (configs.*)
- Factory Pattern
- Deterministic Checks (idempotency/pure functions)
- Reproducibility (Docker) 
- Data transparency / traceability (model payload)


# Wish List:
- CI/CD
- TruncatedSVD
- Multi-vector representations -ColBERT-
- Better pooling strategies
- NMF
- Topic modeling
- Embedding-based similarity search
- More experiments loss function / model payload
- Custom semantic transformer model
- Minimize larger semantic transformer model
- Pure Embedding Clustering
- Graph‑Based Entity Resolution
- LLM as a judge 
- Hybrid Models
- Testing
- Operational Performance (Latency, Cost).
- Model API endpoint, Model card, Prom/Graf/ Pandera/ GE/ Pydantic.
- Stress test model, alert + retraining.
- Model calibration.
- Embedding versioning + reindex cost.
- Optimize model/data for cost/performance.
- Canary deployment

# Data readiness and identity evidence

This notebook begins with the evidence that determines what the reconciliation model may learn. The raw export is never silently filtered: the audit records its exact path and SHA-256 hash, profiles every source column, and retains rows with missing or invalid GTINs.

A structurally valid GTIN is **provisional identity evidence**, not automatically semantic ground truth. A valid GTIN observed at two or more retailers can create cross-source positive evidence. A valid GTIN observed at only one retailer remains a canonical candidate, but cannot verify cross-source matching. Missing or checksum-invalid GTINs remain in the reconciliation corpus and require model-and-gate decisions rather than GTIN-derived labels.

The audit reports a group for review when a valid GTIN has conflicting normalized brand, volume, pack, or category evidence. Title variation is reported separately because it is usually the variation the model must learn, rather than proof that a GTIN is wrong.

Run the audit explicitly whenever the export changes; no filename fallback is allowed:

```bash
MPLCONFIGDIR=/tmp/matplotlib python TRAIN/data_quality_audit.py --input dataset.csv
```


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

audit_dir = Path('results')
audit_paths = {
    'summary': audit_dir / 'data_quality_summary.csv',
    'columns': audit_dir / 'data_quality_columns.csv',
    'groups': audit_dir / 'data_quality_gtin_groups.csv',
}
missing = [str(path) for path in audit_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Data-quality artifacts are missing. Run: '
        'python -m euromonitor.training.data_quality_audit --input dataset.csv\n'
        + '\n'.join(missing)
    )

quality_summary = pd.read_csv(audit_paths['summary'], keep_default_na=False)
quality_columns = pd.read_csv(audit_paths['columns'], keep_default_na=False)
quality_groups = pd.read_csv(audit_paths['groups'], keep_default_na=False)
removed_tokens_path = audit_dir / 'title_removed_tokens.csv'
if not removed_tokens_path.is_file():
    raise FileNotFoundError('Run python -m euromonitor.training.build_title_attribute_evidence to create title_removed_tokens.csv')
removed_tokens = pd.read_csv(removed_tokens_path, keep_default_na=False)

display(quality_summary)
display(quality_columns)


## Ten-row source-to-model trace

This is the inspectable path used by the lane: source text and the two retained evidence fields, canonical evidence, gate outcome, then the exact number-free text supplied to the embedding model. URLs and image URLs are intentionally excluded. The evidence fields are retained for ablation and do not yet alter either gate or model input.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display

sys.path.insert(0, str(Path('src').resolve()))
from euromonitor.pipeline import canonical_model_text, clean_sku_text

results_dir = Path('results')
trace_paths = {
    'canonical': results_dir / 'canonical_records.csv',
    'gates': results_dir / 'gate_results.csv',
}
missing = [str(path) for path in trace_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Run python -m euromonitor.training.data_prep first:\n' + '\n'.join(missing))

raw = pd.read_csv('dataset.csv', dtype=str, keep_default_na=False)
canonical = pd.read_csv(trace_paths['canonical'], dtype={'gtin': str}, keep_default_na=False)
gates = pd.read_csv(trace_paths['gates'], dtype={'gtin1': str, 'gtin2': str}, keep_default_na=False)

# 1. Ten source SKUs, including the two retained-but-not-yet-modeled fields.
source_preview = raw[[
    'sku_id', 'retailer', 'country', 'sku_name_eng', 'description_short_eng',
    'breadcrumbs_eng', 'attribute', 'brand', 'category', 'gtin',
]].head(10).copy()
display(JSON(source_preview.to_dict(orient='records'), expanded=False))

# 2. Canonical records retain description/breadcrumb evidence per valid GTIN.
canonical_preview = canonical[[
    'gtin', 'canonical', 'description_evidence', 'breadcrumb_evidence',
    'volume_set', 'pack_set', 'mode_flavor', 'mode_type',
]].head(10).copy()
canonical_preview['model_canonical_text'] = canonical_preview['canonical'].map(canonical_model_text)
display(JSON(canonical_preview.to_dict(orient='records'), expanded=False))

# 3. The deterministic gate sees canonical text plus extracted package/flavor evidence.
gate_preview = gates[[
    'gtin1', 'gtin2', 'canon1', 'canon2', 'gate_decision', 'gate_reason', 'similarity',
]].head(10)
display(JSON(gate_preview.to_dict(orient='records'), expanded=False))

# 4. Exact SKU-side model text: title + attributes, normalized and number-free.
model_preview = raw[['sku_id', 'sku_name_eng', 'attribute', 'brand']].head(10).copy()
model_preview['model_sku_text'] = model_preview.apply(
    lambda row: clean_sku_text(row['sku_name_eng'], row['attribute'], row['brand']), axis=1
)
display(JSON(model_preview.to_dict(orient='records'), expanded=False))


### Top 10 title tokens removed by shared cleaning

The cleaner records token removal as a multiset difference between normalized original titles and model-cleaned titles.


In [ ]:
display(removed_tokens.head(10))


## GTIN review queue

Prioritize identity contradictions: `brand_conflict`, `volume_conflict`, and `pack_conflict`. These may expose a reused or erroneous GTIN, a parsing error, or different sellable offers sharing one identifier. `category_conflict` is a secondary review signal. `title_variation` is expected retailer variation; sample it for quality assurance instead of treating every differing title as an error. `single_retailer_identity_not_cross_source_verified` is a coverage limitation, not an error.


In [ ]:
priority_reasons = r'brand_conflict|volume_conflict|pack_conflict'
priority_groups = quality_groups.loc[
    quality_groups['review_reasons'].str.contains(priority_reasons, regex=True, na=False)
].copy()

print(f'{len(priority_groups):,} valid-GTIN groups have a brand, volume, or pack contradiction.')
display(
    priority_groups[[
        'gtin', 'rows', 'retailers', 'brands', 'categories',
        'volume_ml', 'pack_count', 'review_reasons', 'titles',
    ]].head(50)
)

reason_counts = (
    quality_groups.loc[quality_groups['review_reasons'].ne(''), 'review_reasons']
    .str.split(' | ', regex=False)
    .explode()
    .value_counts()
    .rename_axis('review_reason')
    .reset_index(name='valid_gtin_groups')
)
display(reason_counts)


## Package-evidence gate verification

Package type and material are live gate constraints. This audit reruns the same compatibility rule against the completed gate decisions: a remaining `proceed → hard_no` row would indicate that package evidence was not applied or that the evidence join drifted.

The table below should be empty after a successful promotion. It retains both evidence sets and the prior gate reason if a gap appears.


In [ ]:
package_impact_paths = {
    'summary': audit_dir / 'package_gate_impact_summary.csv',
    'pairs': audit_dir / 'package_gate_impact_pairs.csv',
}
missing = [str(path) for path in package_impact_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Package-gate impact artifacts are missing. Run: '
        'python -m euromonitor.training.package_gate_impact_audit\n' + '\n'.join(missing)
    )

package_impact_summary = pd.read_csv(package_impact_paths['summary'], keep_default_na=False)
package_impact_pairs = pd.read_csv(package_impact_paths['pairs'], keep_default_na=False)
display(package_impact_summary)

impact_reason_counts = (
    package_impact_pairs['proposed_reason']
    .value_counts()
    .rename_axis('proposed_reason')
    .reset_index(name='pair_count')
)
display(impact_reason_counts)

proceed_to_hard_no = package_impact_pairs.loc[
    package_impact_pairs['old_decision'].eq('proceed')
].copy()
print(f'{len(proceed_to_hard_no):,} current proceed pairs would become hard_no in the shadow rule.')
display(proceed_to_hard_no.head(50))
